╔════════════════════════════════════════════════════════════════════╗
║ 00_build_arrow.py — streaming, memory-safe preprocessing for EPSS ║
║                                                                    ║
║  Produces (in cfg["output_dir"])                                   ║
║    • epss_stage1.arrow  – fully-numeric, sorted by (cve, date)     ║
║    • vocab.json         – categorical-id mapping   (train only)    ║
║    • scaler.pkl         – μ, σ for numeric columns (train only)    ║
╚════════════════════════════════════════════════════════════════════╝

In [1]:
import json, joblib, os, sys
from pathlib import Path

In [2]:
import duckdb, numpy as np, pandas as pd
import pyarrow as pa
import pyarrow.ipc   as ipc           # for integrity check

In [3]:
# ─────────────────────────── configuration ────────────────────────────
local_execution  = False                      # ← flip to False on cluster
EPSS_TRANSFORM   = "inverted_log"            # log | inverted_log | logit | cloglog

In [4]:
LOCAL = {"sample_size": 100_000, "output_dir": "work"}   # dev
CLOUD = {"sample_size": None,     "output_dir": "work"}   # prod
CFG   = LOCAL if local_execution else CLOUD

In [5]:
print(f"[CFG]  {'LOCAL' if local_execution else 'CLOUD'}   "
      f"rows={CFG['sample_size'] or 'ALL'}   "
      f"out=/{CFG['output_dir']}   "
      f"transform={EPSS_TRANSFORM}")

[CFG]  CLOUD   rows=ALL   out=/work   transform=inverted_log


In [6]:
# ─────────────────────────── helpers ──────────────────────────────────
def transform_epss(x: np.ndarray, mode: str = "inverted_log",
                   eps: float = 1e-6) -> np.ndarray:
    p = np.clip(x.astype("float64"), eps, 1.0 - eps)
    if   mode == "log":          out = np.log(p)
    elif mode == "inverted_log": out = -np.log(p)
    elif mode == "logit":        out = np.log(p / (1.0 - p))
    elif mode == "cloglog":      out = np.log(-np.log(1.0 - p))
    else: raise ValueError(mode)
    return out.astype("float32")

In [7]:
# ─────────────────────────── paths ────────────────────────────────────
PARQUET_PATH = (Path("data/full_db/processed/final_full_data.parquet")
                if local_execution else
                Path("../data/final_full_data.parquet"))
OUT_DIR      = Path(CFG["output_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
if not PARQUET_PATH.exists():
    sys.exit(f"[ERR] Parquet not found: {PARQUET_PATH}")

SystemExit: [ERR] Parquet not found: data\final_full_data.parquet

c:\Users\stijn\Desktop\EPSS_FRESH\epss-env\Lib\site-packages\IPython\core\interactiveshell.py:3680: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
SAMPLE_LIM = f" LIMIT {CFG['sample_size']}" if CFG["sample_size"] else ""
print(f"[PATH] parquet={PARQUET_PATH}   out={OUT_DIR}   sample={SAMPLE_LIM or 'FULL'}")

In [ ]:
# ─────────────────────────── constants ────────────────────────────────
DROP_COLS = [
    "description_all", "description_en", "desc_len_all", "details_combined",
    # … (rest of big-text columns) …
    "reference_count"
]
TS_SAFE   = ["published_date"]
TS_LEAKY  = ["last_modified_date", "snapshot_date"]
CAT_COLS  = ["cwe_id", "source_identifier", "vuln_status",
             "canon_severity", "primary_cvss_sev", "prev_event_type"]
BOOL_COLS = lambda cols: [c for c in cols
                          if c.startswith(("has_", "is_"))
                          or c == "same_day_multi_source"]
SENTINEL  = -100.0                           # far outside any z-scored range

In [ ]:
# ─────────────────────────── DuckDB init ──────────────────────────────
parquet_glob = (str(PARQUET_PATH / "*.parquet")
                if PARQUET_PATH.is_dir() else str(PARQUET_PATH))
con = duckdb.connect(database=":memory:")
schema_df = con.execute(f"SELECT * FROM parquet_scan('{parquet_glob}') LIMIT 0").df()

In [ ]:
quoted_drop    = ', '.join(f'"{c}"' for c in DROP_COLS if c in schema_df.columns)
exclude_clause = f" EXCLUDE ({quoted_drop})" if quoted_drop else ""

In [ ]:
row_cnt = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{parquet_glob}'){SAMPLE_LIM}").fetchone()[0]
print(f"[INFO] rows to process (after sample): {row_cnt:,}")

In [ ]:
# ─────────────────────────── train/val/test split dates ───────────────
dates_sql  = f"SELECT DISTINCT date FROM parquet_scan('{parquet_glob}') ORDER BY date"
if CFG["sample_size"]:
    dates_sql += f" LIMIT {CFG['sample_size']}"
unique_dates = con.execute(dates_sql).df()["date"].values
VAL_CUT  = pd.to_datetime(unique_dates[int(0.64 * len(unique_dates))])
TEST_CUT = pd.to_datetime(unique_dates[int(0.80 * len(unique_dates))])
print(f"[SPLIT]  val ≥ {VAL_CUT.date()}   test ≥ {TEST_CUT.date()}")

In [ ]:
# ─────────────────────────── vocabularies (train only) ────────────────
print("[VOCAB] building …")
VOCAB = {}
for col in (c for c in CAT_COLS if c in schema_df.columns):
    vals = con.execute(f"""
        SELECT DISTINCT {col}
        FROM parquet_scan('{parquet_glob}')
        WHERE date < '{VAL_CUT.date()}'
          AND {col} IS NOT NULL
    """).df()[col].dropna().unique()
    cats      = [v for v in vals if v != "UNK"]
    VOCAB[col] = {v: i + 1 for i, v in enumerate(sorted(cats))}
    VOCAB[col]["UNK"] = 0
    print(f"  {col:<22} → {len(cats):>6} categories")

In [ ]:
# ─────────────────────────── numeric μ, σ (train only) ────────────────
NUM_COLS = [c for c, dt in schema_df.dtypes.items()
            if c not in {"cve", "date"} | set(CAT_COLS) and
               ("int" in str(dt).lower() or "float" in str(dt).lower())]
NUM_COLS += [f"{t}_delta" for t in TS_SAFE + TS_LEAKY if t in schema_df.columns]
NUMERIC_ONLY = [c for c in NUM_COLS if c not in {"epss"}]

In [ ]:
print("[STATS] computing μ, σ …")
if NUMERIC_ONLY:
    agg = ", ".join(f"avg({c}) AS mean_{c}, stddev_pop({c}) AS std_{c}"
                    for c in NUMERIC_ONLY)
    ts_deltas = ", ".join(
        f"CASE WHEN {t} IS NOT NULL "
        f"THEN date_diff('day', {t}::date, date::date) END AS {t}_delta"
        for t in TS_SAFE + TS_LEAKY if t in schema_df.columns
    )
    stats_row = con.execute(f"""
        SELECT {agg}
        FROM (
          SELECT *, {ts_deltas}
          FROM parquet_scan('{parquet_glob}')
          WHERE date < '{VAL_CUT.date()}'
        )
    """).fetchone()
    MEAN = {c: float(stats_row[i * 2])     for i, c in enumerate(NUMERIC_ONLY)}
    STD  = {c: float(stats_row[i * 2 + 1]) if stats_row[i * 2 + 1] not in (None, 0) else 1.0
            for i, c in enumerate(NUMERIC_ONLY)}
else:
    MEAN, STD = {}, {}

In [ ]:
# ─────────────────────────── streaming → Arrow IPC ────────────────────
arrow_path = OUT_DIR / "epss_stage1.arrow"
sink       = pa.OSFile(str(arrow_path), "wb")
writer     = None
processed  = 0

In [ ]:
print("[STREAM] writing batches …")
stream_sql = f"""
    SELECT *{exclude_clause}
    FROM parquet_scan('{parquet_glob}')
    ORDER BY cve, date
    {SAMPLE_LIM}
"""

In [ ]:
BATCH_ROWS = 100_000                       # tweak for your RAM / I/O budget

In [ ]:
reader = con.execute(stream_sql) \
             .fetch_record_batch(rows_per_batch=BATCH_ROWS)   # ← ① correct kw

In [ ]:
for rb in reader:                                            # ← ② iterate
    df = rb.to_pandas(use_threads=False)   # zero-copy columns

    # ── feature engineering (unchanged) ───────────────────────────────
    df["date"] = pd.to_datetime(df["date"])

    for t in TS_SAFE:
        if t in df.columns:
            df[f"{t}_delta"] = (df["date"] - pd.to_datetime(df[t])).dt.days
    for t in TS_LEAKY:
        if t in df.columns:
            d = (df["date"] - pd.to_datetime(df[t])).dt.days
            d[d < 0] = np.nan
            df[f"{t}_delta"] = d
    df.drop(columns=[c for c in TS_SAFE + TS_LEAKY if c in df.columns],
            inplace=True)

    df["flag_train"] = (df["date"] <  VAL_CUT).astype("uint8")
    df["flag_val"]   = ((df["date"] >= VAL_CUT) & (df["date"] < TEST_CUT)).astype("uint8")
    df["flag_test"]  = (df["date"] >= TEST_CUT).astype("uint8")

    df["epss"] = transform_epss(df["epss"].values, mode=EPSS_TRANSFORM)

    for col in BOOL_COLS(df.columns):
        if col in df.columns:
            df[col] = df[col].fillna(0).astype("uint8")

    for col, mapping in VOCAB.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(0).astype("int32")

    for col in NUMERIC_ONLY:
        if col in df.columns:
            miss = df[col].isna()
            df[f"{col}_missing"] = miss.astype("uint8")
            df[col] = ((df[col] - MEAN[col]) / STD[col]).astype("float32")
            df.loc[miss, col] = SENTINEL

    # ── write batch ───────────────────────────────────────────────────
    tbl = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pa.ipc.new_file(sink, tbl.schema)
    writer.write(tbl)

    processed += len(df)
    if processed % 500_000 < len(df):          # every ~½ M rows
        print(f"    {processed:>12,} rows")

In [ ]:
writer.close(); sink.close()
print(f"[DONE]  {processed:,} rows → {arrow_path}")

In [ ]:
# ─────────────────────────── save artefacts ──────────────────────────
class StreamingScaler:
    def __init__(self, μ, σ):
        self.mean_  = np.array(list(μ.values()), dtype="float32")
        self.scale_ = np.array(list(σ.values()), dtype="float32")
        self.feature_names_in_ = np.array(list(μ.keys()))
    def transform(self, X):                # X: np.ndarray
        return (X - self.mean_) / self.scale_

In [ ]:
joblib.dump(StreamingScaler(MEAN, STD), OUT_DIR / "scaler.pkl")
json.dump(VOCAB, (OUT_DIR / "vocab.json").open("w"))
print("[SAVE]  vocab.json  scaler.pkl")

In [ ]:
# ─────────────────────────── integrity check ─────────────────────────
print("[CHECK] schema & artefacts …")
tab = ipc.open_file(pa.memory_map(str(arrow_path), "r")).read_all()
assert tab.schema.field("epss").type == pa.float32()
assert len(json.load(open(OUT_DIR / "vocab.json"))) == len(VOCAB)
assert joblib.load(OUT_DIR / "scaler.pkl").mean_.shape[0] == len(MEAN)
print("✓ all artefacts OK")